**Calculate Correlation Coefficient between station observations and incident pixels**


In [1]:
import xarray
import pandas as pd
import glob
import os
import rasterio
import concurrent.futures
from scipy.stats import spearmanr
import os
import numpy as np
from permetrics.regression import RegressionMetric
import HydroErr as he
import hydroeval as hel
import datetime as dt

In [ ]:
datasets = ['CHIRPS', 'PERSIANN', 'ERA5L', 'HAR', 'MSWEP'] 

#Read insitu data
df_insitu = pd.read_csv(r'..\Data\Processed\Insitu\insitu_rainfall_1990_2019.csv')
df_insitu['date'] = pd.to_datetime(df_insitu[['year', 'month', 'day']])

#Get path, files and file list for datasets
def extract_date(file, dataset):
    filename = os.path.basename(file)
    date_str = filename[-12:-4]
    return date_str

def get_dataset_info(dataset):
    dataset_path = fr"..\Data\Datasets\{dataset}_prec"
    dataset_tiff_files = glob.glob(os.path.join(dataset_path, "*.tif"))
    dataset_dates = list(map(lambda file: extract_date(file, dataset), dataset_tiff_files))
    return dataset_path, dataset_tiff_files, dataset_dates

dataset_results = {}
for dataset in datasets:
    dataset_path, dataset_tiff_files, dataset_dates = get_dataset_info(dataset)
    dataset_results[f'{dataset}_path'] = dataset_path
    dataset_results[f'{dataset}_files'] = dataset_tiff_files
    dataset_results[f'{dataset}_dates'] = dataset_dates

#dataset_results['MSWEP_files']

['..\\Data\\Datasets\\MSWEP_prec\\MSWEP_Prec_19900101.tif',
 '..\\Data\\Datasets\\MSWEP_prec\\MSWEP_Prec_19900102.tif',
 '..\\Data\\Datasets\\MSWEP_prec\\MSWEP_Prec_19900103.tif',
 '..\\Data\\Datasets\\MSWEP_prec\\MSWEP_Prec_19900104.tif',
 '..\\Data\\Datasets\\MSWEP_prec\\MSWEP_Prec_19900105.tif',
 '..\\Data\\Datasets\\MSWEP_prec\\MSWEP_Prec_19900106.tif',
 '..\\Data\\Datasets\\MSWEP_prec\\MSWEP_Prec_19900107.tif',
 '..\\Data\\Datasets\\MSWEP_prec\\MSWEP_Prec_19900108.tif',
 '..\\Data\\Datasets\\MSWEP_prec\\MSWEP_Prec_19900109.tif',
 '..\\Data\\Datasets\\MSWEP_prec\\MSWEP_Prec_19900110.tif',
 '..\\Data\\Datasets\\MSWEP_prec\\MSWEP_Prec_19900111.tif',
 '..\\Data\\Datasets\\MSWEP_prec\\MSWEP_Prec_19900112.tif',
 '..\\Data\\Datasets\\MSWEP_prec\\MSWEP_Prec_19900113.tif',
 '..\\Data\\Datasets\\MSWEP_prec\\MSWEP_Prec_19900114.tif',
 '..\\Data\\Datasets\\MSWEP_prec\\MSWEP_Prec_19900115.tif',
 '..\\Data\\Datasets\\MSWEP_prec\\MSWEP_Prec_19900116.tif',
 '..\\Data\\Datasets\\MSWEP_prec\\MSWEP_

In [5]:
#extract station lat and lon in a list 
unique_stn = df_insitu[['index_no','lat','long']].drop_duplicates()
stn_points = list(zip(unique_stn['long'],unique_stn['lat']))
#stn_points
#unique_stn   

In [6]:
def extract_concurrent_grid_data(dataset_results, stn_points, unique_stn):
    df_final = None
    
    for dataset in datasets:
        files = dataset_results[f'{dataset}_files']
        print(f"Processing {dataset} with {len(files)} files...")
        
        def process_file(file):
            date_obj = extract_date(file, dataset) 
            local_records = []
            try:
                with rasterio.open(file) as src:
                    crs = src.crs
                    if crs == "EPSG:4326":
                        sampled_values = list(src.sample(stn_points))
                        for i, value in enumerate(sampled_values):
                            local_records.append({
                                'date': date_obj,
                                'stn_id': unique_stn.iloc[i]['index_no'],
                                'prec_grid': value[0]
                                })
            except Exception as e:
                print(f'Error processing {file}: {e}')
            return local_records

        dataset_records = []
        with concurrent.futures.ThreadPoolExecutor() as executor:
            results = executor.map(process_file, files)
            for result in results:
                dataset_records.extend(result)
        
        df_dataset = pd.DataFrame(dataset_records)
        
        # CHECK IF EMPTY FIRST
        if not df_dataset.empty:
            df_dataset = df_dataset[df_dataset['prec_grid'] != -9999.0]
            df_dataset = df_dataset.dropna().round(1)
            df_dataset.loc[df_dataset['prec_grid'] < 0, 'prec_grid'] = 0
            df_dataset = df_dataset.rename(columns={'prec_grid': f'{dataset}_prec'})
            
            # Merge logic
            if df_final is None:
                df_final = df_dataset
            else:
                df_final = pd.merge(df_final, df_dataset, on=['date', 'stn_id'], how='inner')
                print(f"Merged {dataset}. Resulting shape: {df_final.shape}")
        else:
            print(f"Warning: No data extracted for {dataset}")

    return df_final

df_grid_data = extract_concurrent_grid_data(dataset_results, stn_points, unique_stn)

Processing CHIRPS with 10957 files...
Processing PERSIANN with 10957 files...
Merged PERSIANN. Resulting shape: (936282, 4)
Processing ERA5L with 10956 files...
Merged ERA5L. Resulting shape: (936196, 5)
Processing HAR with 10957 files...
Merged HAR. Resulting shape: (936196, 6)
Processing MSWEP with 10957 files...
Merged MSWEP. Resulting shape: (936196, 7)


In [7]:
#merging station and grid df into a single df
df_insitu1 = df_insitu[['date','index_no','rainfall_sum']]
df_insitu1 = df_insitu1.rename(columns={'rainfall_sum':'prec_insitu', 'index_no':'stn_id'})

df_grid_data['date'] = pd.to_datetime(df_grid_data['date'])
df_insitu1['date'] = pd.to_datetime(df_insitu1['date'])
df_merged = pd.merge(df_grid_data, df_insitu1, on=['date','stn_id'], how='inner')
df_merged.to_csv(r'../Data/Processed/Accuracy_assessment/precipitation_daily_merged.csv', index=False)

**Calculating error metrices & CPI- Daily Scale**

In [8]:
df_merged = pd.read_csv(r'../Data/Processed/Accuracy_assessment/precipitation_daily_merged.csv')

#column normalization function
def normalize_column(col):
    return (col - col.min()) / (col.max() - col.min())
          
#calculate accuracy of datasets
def calculate_accuracy(dataframe_prec):
    df_accuracy = pd.DataFrame()

    #datasets = ['ERA5L', 'HARv2', 'CHIRPS', 'MSWEP', 'PERSIANN']

    for dataset in datasets:
        dataset_accuracy = {'dataset': dataset}

        #Convert to array
        prec_grid_array = dataframe_prec[f'{dataset}_prec'].values
        prec_insitu_array = dataframe_prec['prec_insitu'].values
    
        #drop null values
        valid_indices = np.logical_and(~np.isnan(prec_grid_array), ~np.isnan(prec_insitu_array))
        observations = np.sum(valid_indices)
        dataset_accuracy['observations_n'] = observations

        prec_grid_array = prec_grid_array[valid_indices]
        prec_insitu_array = prec_insitu_array[valid_indices]

        #correlation coefficient
        correlation = he.spearman_r(prec_grid_array, prec_insitu_array)
        dataset_accuracy['correlation'] = correlation

        #NRMSE
        nrmse = he.nrmse_mean(prec_grid_array, prec_insitu_array)
        dataset_accuracy['nrmse'] = nrmse

        #Percent bias
        pbias = 100 * np.sum(prec_grid_array - prec_insitu_array) / np.sum(prec_insitu_array)
        dataset_accuracy['pbias'] = pbias

        #Relative standard deviation 
        rsd = (np.std(prec_grid_array))/(np.std(prec_insitu_array))
        dataset_accuracy['rsd'] = rsd

        df_dataset_accuracy = pd.DataFrame([dataset_accuracy])
        df_accuracy = pd.concat([df_accuracy, df_dataset_accuracy], ignore_index=True)

    #Normalization
    df_accuracy['abs_pbias'] = df_accuracy['pbias'].abs()
    df_accuracy['diff_rsd'] = (df_accuracy['rsd']-1).abs()

    df_accuracy_normalized = df_accuracy.copy().drop(columns=['pbias'])

    metrics = ['correlation', 'nrmse', 'abs_pbias', 'diff_rsd']
    for column in metrics:
        df_accuracy_normalized[column] = normalize_column(df_accuracy_normalized[column])

    #converting to scale of 0 to 1 (ideal to worst) - correlation & rsd needs inversion
    df_accuracy_normalized['correlation'] = 1 - df_accuracy_normalized['correlation']

    #get cpi for each (linear)
    df_accuracy_normalized['cpi'] = df_accuracy_normalized['correlation'] + df_accuracy_normalized['nrmse'] + df_accuracy_normalized['abs_pbias'] + df_accuracy_normalized['diff_rsd']
    df_accuracy_normalized['cpi_rank'] = df_accuracy_normalized['cpi'].rank(ascending=True)

    df_accuracy_final = df_accuracy_normalized[['dataset','cpi','cpi_rank']].merge(df_accuracy[['dataset','correlation','nrmse','pbias','rsd', 'observations_n']], on=['dataset'], how='inner')
    return df_accuracy_final

#get daily data accuracy
df_accuracy_daily = calculate_accuracy(df_merged)
df_accuracy_daily

,dataset,cpi,cpi_rank,correlation,nrmse,pbias,rsd,observations_n
0,CHIRPS,2.061630,4.0,0.279015,3.433587,-11.235871,0.936553,706296
1,PERSIANN,1.266833,3.0,0.416261,2.944956,-29.541271,0.573728,706296
2,ERA5L,0.942888,1.0,0.579364,3.240967,35.544650,1.033149,706296
3,HAR,1.049144,2.0,0.531979,3.350039,16.107354,1.011180,706296
4,MSWEP,2.579216,5.0,0.432026,2.988277,-89.352765,0.140749,706296


**Accuracy and CPI Calculation - Seasonal Scale**

In [11]:
#define seasons
seasons = {
    'winter': [12, 1, 2],
    'pre_monsoon': [3, 4, 5],
    'monsoon': [6, 7, 8, 9],
    'post_monsoon': [10, 11]
}
df_merged1 = df_merged.copy()
df_merged1['date'] = pd.to_datetime(df_merged1['date'])
df_merged1['year'] = df_merged1['date'].dt.year
df_merged1['month'] = df_merged1['date'].dt.month

#create effective year for seasonal analysis
df_merged1['year_seasonal'] = df_merged1['year']
df_merged1.loc[df_merged1['month'] == 12, 'year_seasonal'] = df_merged1.loc[df_merged1['month'] == 12, 'year'] + 1

df_merged1['season'] = (df_merged1['month'].apply(lambda x: [k for k, v in seasons.items() if x in v][0]))

#calculate seasonal rainfall sum for each dataset 
df_seasonal = df_merged1.groupby(by = ['season','year_seasonal'])[['CHIRPS_prec',	'PERSIANN_prec', 'ERA5L_prec', 'prec_insitu','HAR_prec', 'MSWEP_prec']].sum().reset_index()
df_seasonal

,season,year_seasonal,CHIRPS_prec,PERSIANN_prec,ERA5L_prec,prec_insitu,HAR_prec,MSWEP_prec
0,monsoon,1990,107874.7,82539.0,148626.0,83683.9,109828.6,12900.8
1,monsoon,1991,84037.6,76083.1,122343.1,78672.7,104115.4,9021.9
2,monsoon,1992,63587.9,64154.1,97761.2,65389.0,85002.7,8041.3
3,monsoon,1993,85311.1,94388.8,146120.3,78354.6,106609.8,10896.6
4,monsoon,1994,88944.2,77367.0,121043.8,75957.2,97969.4,13897.1
...,...,...,...,...,...,...,...,...
116,winter,2016,2156.7,25781.6,8128.2,1547.3,3679.3,271.5
117,winter,2017,2540.1,1988.2,6087.4,1385.3,4214.2,237.3
118,winter,2018,2585.7,1696.8,5027.6,1146.9,2639.8,149.6
119,winter,2019,8271.2,8354.9,15826.5,7464.1,17361.4,1012.5


In [12]:
#caluclate overall seasonal cpi
df_accuracy_seasonal = (calculate_accuracy(df_seasonal))
df_accuracy_seasonal['season'] = 'all'
#print("OVERALL SEASONAL ACCURACY:")
#print(df_accuracy_seasonal)

#calculate monsoonal cpi
df_monsoon = df_seasonal[df_seasonal['season'] == 'monsoon']
df_accuracy_monsoon = calculate_accuracy(df_monsoon)
df_accuracy_monsoon['season'] = 'monsoon'
#print("MONSOON ACCURACY:")
#print(df_accuracy_monsoon)

#calculate pre-monsoon cpi
df_premonsoon = df_seasonal[df_seasonal['season'] == 'pre_monsoon']
df_accuracy_premonsoon = calculate_accuracy(df_premonsoon)
df_accuracy_premonsoon['season'] = 'premonsoon'
#print("PRE-MONSOON ACCURACY:")
#print(df_accuracy_premonsoon)

#calculate post-monsoon cpi
df_postmonsoon = df_seasonal[df_seasonal['season'] == 'post_monsoon']
df_accuracy_postmonsoon = calculate_accuracy(df_postmonsoon)
df_accuracy_postmonsoon['season'] = 'postmonsoon'
#print("POST-MONSOON ACCURACY:")
#print(df_accuracy_postmonsoon)

#concatenate all cpi
df_accuracy_final = pd.concat([df_accuracy_seasonal, df_accuracy_monsoon, df_accuracy_premonsoon, df_accuracy_postmonsoon])
df_accuracy_final

,dataset,cpi,cpi_rank,correlation,nrmse,pbias,rsd,observations_n,season
0,CHIRPS,1.289681,3.0,0.954491,0.389232,18.667277,1.202135,121,all
1,PERSIANN,0.958485,1.0,0.955575,0.323286,-7.079361,0.857332,121,all
2,ERA5L,2.712353,4.0,0.976080,1.237648,78.225803,1.720414,121,all
3,HAR,1.287446,2.0,0.980599,0.752695,50.479884,1.391040,121,all
4,MSWEP,3.638298,5.0,0.963934,1.415663,-85.911999,0.134921,121,all
0,CHIRPS,1.151062,2.0,0.362848,0.238411,19.628149,1.324750,30,monsoon
1,PERSIANN,0.791736,1.0,0.415795,0.191284,-13.238323,1.320218,30,monsoon
2,ERA5L,2.767592,4.0,0.600000,0.756972,73.733317,2.091330,30,monsoon
3,HAR,1.515425,3.0,0.628921,0.442757,41.250871,1.907891,30,monsoon
4,MSWEP,3.660438,5.0,0.359733,0.870008,-86.415177,0.170510,30,monsoon


**Accuracy and CPI Calculation - Annual Scale**

In [14]:
df_merged2 = df_merged.copy()
df_merged2['date'] = pd.to_datetime(df_merged2['date'])
df_merged2['year'] = df_merged2['date'].dt.year
df_annual = df_merged2.groupby(by = ['year'])[['CHIRPS_prec',	'PERSIANN_prec', 'ERA5L_prec', 'prec_insitu', 'HAR_prec', 'MSWEP_prec']].sum().reset_index()
df_accuracy_annual = calculate_accuracy(df_annual)
df_accuracy_annual


,dataset,cpi,cpi_rank,correlation,nrmse,pbias,rsd,observations_n
0,CHIRPS,1.089522,2.0,0.430923,0.228464,18.667277,1.410893,30
1,PERSIANN,0.273973,1.0,0.588877,0.134879,-7.079361,1.332286,30
2,ERA5L,2.806446,4.0,0.695662,0.794469,78.225803,1.914341,30
3,HAR,1.893399,3.0,0.662291,0.521754,50.479884,1.755465,30
4,MSWEP,3.879726,5.0,0.305895,0.864557,-85.911999,0.155665,30
